# Financial Research Agent — Notebook Demo
Quick experimentation without the UI.

In [ ]:
import os, sys
sys.path.append('..')
os.environ['OPENAI_API_KEY'] = 'your-key-here'

from src.rag_pipeline import FinancialRAGPipeline
from src.agent import FinancialResearchAgent
from src.summarizer import FinancialSummarizer

In [ ]:
# ── Load a document ──
rag = FinancialRAGPipeline()

# Option A: Load PDF
# docs = rag.load_pdf('../data/apple_10k_2024.pdf')

# Option B: Load from URL (SEC EDGAR)
docs = rag.load_from_url('https://www.sec.gov/cgi-bin/browse-edgar?action=getcompany&CIK=AAPL&type=10-K')

docs = rag.add_metadata(docs, doc_type='10-K', company='Apple', year='2024')
rag.ingest(docs)
print(f'Loaded {len(docs)} pages')

In [ ]:
# ── Ask a question ──
agent = FinancialResearchAgent(rag=rag)
result = agent.run("What are Apple's main revenue segments and how did they perform?")
print(result['answer'])

In [ ]:
# ── Generate full investment memo ──
summarizer = FinancialSummarizer()
docs_ctx = rag.retrieve('Apple financials revenue risks guidance', k=10)
context = '\n\n'.join([d.page_content for d in docs_ctx])
memo = summarizer.generate_investment_memo(company='Apple', context=context)
print(summarizer.format_report_markdown(memo))